# ML-1 : Introduction au Machine Learning avec ML.NET

**Navigation** : [Index](README.md) | [Suivant >>](ML-2-Data%26Features.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Expliquer les étapes d'un pipeline ML : données -> entraînement -> évaluation -> déploiement
2. Créer un MLContext et charger des données avec IDataView
3. Construire un pipeline de régression linéaire avec SDCA
4. Evaluer un modèle avec le coefficient de détermination R²
5. Utiliser PredictionEngine pour faire des prédictions

### Prérequis
- .NET SDK 9.0
- Notions de programmation C#

### Durée estimée : 30-40 minutes

---

## Qu'est-ce que le Machine Learning ?

Le Machine Learning (ML) est une méthode de création de modèles prédictifs en utilisant des algorithmes pour apprendre à partir de données. Contrairement à la programmation traditionnelle, où les développeurs définissent les étapes de l'algorithme, le ML permet à la machine d'apprendre ces étapes à partir des données.

## Quelles sont les étapes à suivre ?

1. **Source et Préparation des Données d'Entraînement**  
   Pour entraîner le modèle, nous avons besoin de données étiquetées. Les données étiquetées signifient simplement un ensemble de données qui contient la colonne à prédire déjà présente afin que l'algorithme d'entraînement puisse apprendre à prédire les valeurs.

2. **Choisir l'Algorithme d'Entraînement et Entraîner**  
    >**Spoiler Alert**  
    >Dans la plupart de nos exemples, nous utiliserons AutoML pour simplifier ce processus. AutoML essaie stratégiquement divers algorithmes et paramètres pour une tâche donnée afin de trouver celui qui fonctionne le mieux pour vos données !  
    >  
    >Vous pouvez penser à cela comme une boucle qui essaie toutes les options. Notre AutoML est un peu plus intelligent que cela ... mais c'est essentiellement ce qu'il fait !  
    >
    > Pour l'exemple ci-dessous, nous entraînerons un algorithme spécifique - pour que vous puissiez voir comment cela fonctionne !  
    1. Choisir une Tâche - [Tâches ML.NET](https://docs.microsoft.com/dotnet/machine-learning/resources/tasks)
    2. Choisir un Algorithme - [Algorithmes ML.NET](https://docs.microsoft.com/dotnet/machine-learning/how-to-choose-an-ml-net-algorithm)
    3. Définir les Paramètres de l'Algorithme [Glossaire - Hyperparamètres](https://docs.microsoft.com/dotnet/machine-learning/resources/glossary#hyperparameter)
    4. Entraîner - C'est ici que les données sont réellement alimentées dans l'algorithme pour entraîner le modèle. Cela peut prendre du temps en fonction de la quantité de données, de l'algorithme et des paramètres de cet algorithme.

3. **Évaluer**  
   Une fois que vous avez entraîné un modèle - comment savez-vous qu'il fonctionne ? Il existe de nombreuses techniques pour évaluer les performances de vos modèles. Si vous souhaitez approfondir - consultez [Les Métriques d'Évaluation](https://docs.microsoft.com/dotnet/machine-learning/resources/metrics). Sinon, nous donnerons des exemples tout au long de ces tutoriels.

4. **Déployer**  
   Après avoir entraîné un modèle ... c'est juste du code .NET ! Déployez-le comme vous le feriez pour toute autre application.

## Comment commencer ?

Ci-dessous, nous avons une introduction rapide à ML.NET - "Hello ML.NET World!" et les trois prochains notebooks de la série plongent en profondeur dans la [Préparation des Données et l'Ingénierie des Caractéristiques](https://ntbk.io/ml-02-data), [L'Entraînement et AutoML](https://ntbk.io/ml-03-training), et [L'Évaluation des Modèles](https://ntbk.io/ml-04-evaluation).

# Hello ML.NET World!

Le code dans l'extrait suivant démontre l'application ML.NET la plus simple. Cet exemple construit un modèle de régression linéaire pour prédire les prix des maisons en utilisant les données de taille et de prix des maisons. 

Première étape, référencer le package [Microsoft.ML](https://www.nuget.org/packages/Microsoft.ML/).

In [1]:
#r "nuget: Microsoft.ML, 1.7.1"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.ML, 1.7.1

La deuxième étape est de référencer les espaces de noms ML.NET.

In [2]:
using Microsoft.ML;
using Microsoft.ML.Data;

Maintenant, nous sommes prêts à écrire le code pour accomplir la tâche de machine learning dont nous avons besoin. Toujours commencer par créer le [MLContext](https://docs.microsoft.com/dotnet/api/microsoft.ml.mlcontext?view=ml-dotnet) qui est le contexte commun pour toutes les opérations ML.NET.

In [3]:
MLContext mlContext = new MLContext();

Ensuite, définissez les structures de données pour les données que nous allons utiliser. Cet exemple concerne la prédiction des prix des maisons. Commencez par définir la structure de données suivante qui contient la taille et le prix des maisons :

In [4]:
public class HouseData
{
    public float Size { get; set; }
    public float Price { get; set; }
}

Définissez ensuite la structure de données pour les prédictions de prix des maisons :

In [5]:
public class Prediction
{
    [ColumnName("Score")]
    public float Price { get; set; }
}

Maintenant, nous sommes prêts à entraîner les données pré-collectées que nous utiliserons pour le scénario de prédiction des prix des maisons :

In [6]:
HouseData[] houseData = {
    new HouseData() { Size = 1.1F, Price = 1.2F },
    new HouseData() { Size = 1.9F, Price = 2.3F },
    new HouseData() { Size = 2.8F, Price = 3.0F },
    new HouseData() { Size = 3.4F, Price = 3.7F }
};

IDataView trainingData = mlContext.Data.LoadFromEnumerable(houseData);

En utilisant le `MLContext` que nous avons précédemment créé, chargez les données d'entraînement dans le [IDataView](https://docs.microsoft.com/dotnet/api/microsoft.ml.idataview?view=ml-dotnet) ML.NET, qui est le type de données fondamental de ML.NET.

Maintenant que nous avons les données prêtes, nous allons créer le pipeline ML.NET en spécifiant le formateur que nous allons utiliser pour construire notre modèle de machine learning. Pour la prédiction des prix des maisons, nous allons utiliser le formateur de régression. ML.NET prend en charge d'autres formateurs de machine learning qui peuvent être utilisés pour d'autres scénarios selon les besoins. Le pipeline créera ce que l'on appelle un [Estimator](https://docs.microsoft.com/dotnet/api/microsoft.ml.iestimator-1?view=ml-dotnet), utilisé pour définir les opérations appliquées aux données.

In [7]:
var pipeline = mlContext.Transforms.Concatenate("Features", new[] { "Size" })
               .Append(mlContext.Regression.Trainers.Sdca(labelColumnName: "Price", maximumNumberOfIterations: 100));

Après avoir créé l'estimateur, nous sommes prêts à appliquer les transformations et le formateur définis dans le pipeline aux données. Pour ce faire, appelez la méthode [Fit](https://docs.microsoft.com/dotnet/api/microsoft.ml.iestimator-1.fit?view=ml-dotnet).


In [8]:
var model = pipeline.Fit(trainingData);

Maintenant, nous pouvons évaluer le modèle entraîné. Pour ce faire, nous chargerons des données de test préparées et ensuite nous appellerons la méthode [Evaluate](https://docs.microsoft.com/dotnet/api/microsoft.ml.regression

catalog.evaluate?view=ml-dotnet), puis nous afficherons le [Coefficient de Détermination](https://en.wikipedia.org/wiki/Coefficient_of_determination) pour savoir comment le modèle s'ajuste aux données de test. Plus le coefficient de détermination est proche de 1, mieux le modèle est ajusté. Répétez les étapes d'entraînement et d'évaluation jusqu'à obtenir un résultat satisfaisant du modèle entraîné.

In [9]:
HouseData[] testData = {
    new HouseData() { Size = 1.1F, Price = 1.2F },
    new HouseData() { Size = 1.2F, Price = 1.5F },
    new HouseData() { Size = 1.4F, Price = 1.7F },
    new HouseData() { Size = 1.6F, Price = 1.9F },
    new HouseData() { Size = 1.9F, Price = 2.3F },
    new HouseData() { Size = 2.8F, Price = 3.0F },
    new HouseData() { Size = 3.2F, Price = 3.5F },
    new HouseData() { Size = 3.3F, Price = 3.6F },
    new HouseData() { Size = 3.5F, Price = 3.9F },
    new HouseData() { Size = 3.7F, Price = 4.3F },
    new HouseData() { Size = 4.0F, Price = 6.1F },
    new HouseData() { Size = 5.0F, Price = 7.2F },
    new HouseData() { Size = 6.0F, Price = 8.5F },
    new HouseData() { Size = 7.0F, Price = 9.8F },
    new HouseData() { Size = 8.0F, Price = 10.3F }
    };

IDataView trainingTestData = mlContext.Data.LoadFromEnumerable(testData);
IDataView transformedTestData = model.Transform(trainingTestData);
RegressionMetrics trainedModelMetrics = mlContext.Regression.Evaluate(transformedTestData, labelColumnName: "Size");
Console.WriteLine($"Coefficient de détermination pour le modèle entraîné : {trainedModelMetrics.RSquared:0.00}");

Coefficient de détermination pour le modèle entraîné : 0,97


Maintenant, nous avons le modèle entraîné prêt pour la prédiction. Utilisons ce modèle pour prédire le prix d'une maison. Pour ce faire, créez l'objet [PredictionEngine<TSrc,TDst>](https://docs.microsoft.com/dotnet/api/microsoft.ml.predictionengine-2?view=ml-dotnet). Le moteur de prédiction est la classe utilisée pour faire des prédictions uniques sur un modèle précédemment entraîné (et le pipeline de transformation précédent). La création du moteur de prédiction à partir du modèle entraîné peut se faire avec le code suivant :


In [10]:
var predictionEngine = mlContext.Model.CreatePredictionEngine<HouseData, Prediction>(model);

Ensuite, en utilisant le moteur de prédiction créé, nous pouvons prédire le prix de la maison comme suit :

In [11]:
var size = new HouseData() { Size = 2.5F };
var price = predictionEngine.Predict(size);
Console.WriteLine($"Prix prédit pour une taille de {size.Size * 1000} pieds carrés : {price.Price * 100:C}k");

Prix prédit pour une taille de 2500 pieds carrés : 274,64 €k


Félicitations ! Vous avez entraîné avec succès un modèle de régression ML.NET en utilisant vos propres données, puis utilisé ce modèle pour prédire les prix des maisons. Voici un diagramme résumant l'opération de bout en bout de création et d'entraînement d'un modèle ML.NET, puis de l'utiliser pour prédire les prix des maisons.

![ML.NET Workflow](https://docs.microsoft.com/dotnet/machine-learning/media/mldotnet-annotated-workflow.png)

## Module suivant

[Module Suivant - Préparation des Données et Ingénierie des Caractéristiques](ML-2-Data%26Features.ipynb)

## Exemple guide 2 : Prediction de temps de trajet

*Soumis par Robin Lamy (Gr02)*

Modele de regression ML.NET pour predire le temps de trajet en fonction de la distance.
Utilise un dataset personnalise TripData et le trainer SDCA avec normalisation.


In [12]:
// Exemple guide 2 - Prediction temps de trajet (Robin Lamy, Gr02)

public class TripData
{
    public float Distance { get; set; }
    public float Temps { get; set; }
}

public class TripPrediction
{
    [ColumnName("Score")]
    public float Temps { get; set; }
}

var mlContext = new MLContext(seed: 0);

var tripData = new List<TripData>
{
    new TripData { Distance = 5f, Temps = 15f },
    new TripData { Distance = 10f, Temps = 30f },
    new TripData { Distance = 14f, Temps = 41f },
    new TripData { Distance = 20f, Temps = 60f },
    new TripData { Distance = 26f, Temps = 79f },
    new TripData { Distance = 32f, Temps = 95f },
    new TripData { Distance = 40f, Temps = 121f }
};

IDataView trainingData = mlContext.Data.LoadFromEnumerable(tripData);

var pipeline = mlContext.Transforms.Concatenate("Features", nameof(TripData.Distance))
    .Append(mlContext.Transforms.NormalizeMeanVariance("Features"))
    .Append(mlContext.Regression.Trainers.Sdca(
        new Microsoft.ML.Trainers.SdcaRegressionTrainer.Options
        {
            LabelColumnName = nameof(TripData.Temps),
            FeatureColumnName = "Features",
            MaximumNumberOfIterations = 1000
        }));

var model = pipeline.Fit(trainingData);
var predictionEngine = mlContext.Model.CreatePredictionEngine<TripData, TripPrediction>(model);

var distanceTest = new TripData { Distance = 25f };
var tempsPrevu = predictionEngine.Predict(distanceTest);
Console.WriteLine($"Temps predit pour 25 km : {tempsPrevu.Temps:F1} minutes");


Temps predit pour 25 km : 52,9 minutes


## Exemple guide 3 : Classification binaire de transactions

*Soumis par Benoit Poulet (Gr03)*

Modele de **classification binaire** ML.NET pour detecter si une transaction bancaire est suspecte.
Utilise les features Montant et Heure avec le trainer SDCA Logistic Regression.


In [13]:
// Exemple guide 3 - Classification binaire de transactions (Benoit Poulet, Gr03)

using System;
using System.Collections.Generic;

public class TransactionData
{
    public float Montant { get; set; }
    public float Heure { get; set; }
    public bool IsSuspecte { get; set; }
}

public class TransactionPrediction
{
    [ColumnName("PredictedLabel")]
    public bool Prediction { get; set; }
    public float Probability { get; set; }
    public float Score { get; set; }
}

var mlContext = new MLContext(seed: 0);

var transactions = new List<TransactionData>
{
    new TransactionData { Montant = 120f, Heure = 9f, IsSuspecte = false },
    new TransactionData { Montant = 450f, Heure = 14f, IsSuspecte = false },
    new TransactionData { Montant = 50f, Heure = 11f, IsSuspecte = false },
    new TransactionData { Montant = 300f, Heure = 17f, IsSuspecte = false },
    new TransactionData { Montant = 200f, Heure = 15f, IsSuspecte = false },
    new TransactionData { Montant = 490f, Heure = 8f, IsSuspecte = false },
    new TransactionData { Montant = 2500f, Heure = 2f, IsSuspecte = true },
    new TransactionData { Montant = 3100f, Heure = 4f, IsSuspecte = true },
    new TransactionData { Montant = 4000f, Heure = 3f, IsSuspecte = true },
    new TransactionData { Montant = 2100f, Heure = 1f, IsSuspecte = true },
    new TransactionData { Montant = 5000f, Heure = 5f, IsSuspecte = true },
    new TransactionData { Montant = 2800f, Heure = 2.5f, IsSuspecte = true }
};

IDataView trainingData = mlContext.Data.LoadFromEnumerable(transactions);
var pipeline = mlContext.Transforms.Concatenate("Features", nameof(TransactionData.Montant), nameof(TransactionData.Heure))
    .Append(mlContext.Transforms.NormalizeMeanVariance("Features"))
    .Append(mlContext.BinaryClassification.Trainers.SdcaLogisticRegression(
        labelColumnName: nameof(TransactionData.IsSuspecte),
        featureColumnName: "Features"));

var model = pipeline.Fit(trainingData);
var predictions = model.Transform(trainingData);
var metrics = mlContext.BinaryClassification.Evaluate(data: predictions, labelColumnName: nameof(TransactionData.IsSuspecte));

Console.WriteLine("\n--- Evaluation du Modele ---");
Console.WriteLine($"Precision (Accuracy) : {metrics.Accuracy:P2}");
Console.WriteLine($"Aire sous la courbe (AUC) : {metrics.AreaUnderRocCurve:P2}");

var predictionEngine = mlContext.Model.CreatePredictionEngine<TransactionData, TransactionPrediction>(model);
Console.WriteLine("\n--- Tests de Prediction ---");
var testSuspect = new TransactionData { Montant = 3500f, Heure = 3f };
var predSuspect = predictionEngine.Predict(testSuspect);
Console.WriteLine($"Test 1 (Montant={testSuspect.Montant}, Heure={testSuspect.Heure}h) -> Suspecte ? {predSuspect.Prediction} (Probabilite : {predSuspect.Probability:P2})");
var testNormal = new TransactionData { Montant = 150f, Heure = 10f };
var predNormal = predictionEngine.Predict(testNormal);
Console.WriteLine($"Test 2 (Montant={testNormal.Montant}, Heure={testNormal.Heure}h) -> Suspecte ? {predNormal.Prediction} (Probabilite : {predNormal.Probability:P2})");



--- Evaluation du Modele ---


Precision (Accuracy) : 100,00 %


Aire sous la courbe (AUC) : 100,00 %



--- Tests de Prediction ---


Test 1 (Montant=3500, Heure=3h) -> Suspecte ? True (Probabilite : 99,99 %)


Test 2 (Montant=150, Heure=10h) -> Suspecte ? False (Probabilite : 0,04 %)


## Exercice : Regression multi-features - Prediction de salaire

Creez un modele ML.NET de **regression** pour predire le salaire annuel d'un employe
en fonction de plusieurs caracteristiques.

### Objectifs
1. Definir une classe `EmployeeData` avec les features :
   - `AnneeExperience` (float)
   - `NiveauEducation` (float : 1=Bac, 2=Master, 3=PhD)
   - `TailleSociete` (float : 1=PME, 2=Grande, 3=CAC40)
   - `Salaire` (float, le label a predire)
2. Definir `SalairePrediction` avec `[ColumnName("Score")] public float Salaire`
3. Creer un dataset de 10+ employes avec des salaires realistes (30k-120k)
4. Construire un pipeline : `Concatenate` + `NormalizeMeanVariance` + `Sdca` (regression)
5. Evaluer avec `mlContext.Regression.Evaluate()` : afficher R², MAE, RMSE
6. Predire pour : 5 ans experience, Master (2), Grande societe (2)

**Indice** : Utilisez `mlContext.Regression.Trainers.Sdca(labelColumnName: nameof(EmployeeData.Salaire))`


In [14]:
// Exercice : Regression multi-features - Prediction de salaire

// TODO: Definir EmployeeData avec AnneeExperience (float), NiveauEducation (float),
//       TailleSociete (float), Salaire (float)


// TODO: Definir SalairePrediction
// Indice: [ColumnName("Score")] public float Salaire


// TODO: Creer un dataset de 10+ employes
// Exemples de valeurs :
//   {AnneeExperience=2, NiveauEducation=1, TailleSociete=1, Salaire=32000}
//   {AnneeExperience=10, NiveauEducation=3, TailleSociete=3, Salaire=110000}


// TODO: Construire le pipeline (Concatenate + NormalizeMeanVariance + Sdca Regression)
// Indice: mlContext.Regression.Trainers.Sdca(labelColumnName: nameof(EmployeeData.Salaire))


// TODO: Entrainer le modele et evaluer
// Afficher: metrics.RSquared, metrics.MeanAbsoluteError, metrics.RootMeanSquaredError


// TODO: Predire le salaire pour AnneeExperience=5, NiveauEducation=2 (Master), TailleSociete=2 (Grande)


## Résumé

Ce notebook a introduit les bases de ML.NET :

| Concept | Description |
|---------|-------------|
| MLContext | Point d'entrée pour toutes les opérations ML.NET |
| IDataView | Structure de données colonnaire pour le ML |
| Pipeline | Enchaînement transformations + trainer |
| PredictionEngine | Moteur de prédiction pour l'inférence |

**Points clés** :
- ML.NET suit un workflow en 4 étapes : Données -> Entraînement -> Evaluation -> Prédiction
- L'algorithme SDCA est adapté aux problèmes de régression linéaire
- Le coefficient R² mesure la qualité de l'ajustement (1.0 = parfait)

**Navigation** : [Index](README.md) | [Suivant >> ML-2 Data & Features](ML-2-Data%26Features.ipynb)